In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A4

kt : ---> VDD3V3

ks : ---> relay.ch1 (VIN)

dm.ch3 : ---> relay.ch3 and GND (R_EN)

ps.ch1 : relay

relay.ch1 : VIN

relay.ch3 : EN

loader : disconnect

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_4] EN Pin R_PD")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "R_EN (ohm)"])

In [ ]:
disable_all_ps()
relay.init_channels

list_vin = [5, 20, 28]
v_vdd = 3.3

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

relay.ch3.enable # en
delay(0.5)

ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

dm.ch1.current_20mA
relay.ch1.enable # vin
delay(0.5)

In [ ]:
# ----------------------------------------------------------------------------

for vin in list_vin:
    
    ks.vset = vin
    
    r_en = dm.ch3.resistance
    ret = [vin, v_vdd, r_en]
    log.output_csv(ret)
        
    print(ret)

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()